<a href="https://colab.research.google.com/github/StrawEater/PracticasPDI3erBimestre/blob/main/Tareas/Tarea_2/Tarea2_Deconvolucion_PDI2026_Realizar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# El Teorema de Convolución en Acción

**Procesamiento de Imágenes**

Ya conocemos la Transformada de Fourier (TF), su inversa, y sus versiones discreta (DFT) y bidimensional (DFT-2D). También conocemos varios kernels de convolución: la delta de Kronecker, el kernel Gaussiano, y los operadores de gradiente Sobel, Prewitt y Roberts.

Lo que todavía no vimos es **cómo se relacionan la convolución y la Transformada de Fourier**.

> **Teorema de convolución.** Si $f$ y $h$ son dos señales (o imágenes) y $*$ denota la convolución, entonces
> $$ \mathcal{F}\{f * h\} = \mathcal{F}\{f\} \cdot \mathcal{F}\{h\} $$
> Es decir: **convolucionar en el dominio espacial equivale a multiplicar, punto a punto, en el dominio de la frecuencia.**
>
> De forma equivalente, aplicando la TF inversa:
> $$ f * h = \mathcal{F}^{-1}\{\mathcal{F}\{f\} \cdot \mathcal{F}\{h\}\} $$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import convolve2d  # SOLO como referencia para comparar/verificar, no es el "método nuevo" que vamos a construir
from skimage import data, img_as_float

plt.rcParams['figure.facecolor'] = 'white'


def preparar_kernel(kernel, forma):
    '''
    Ubica un kernel chico (por ejemplo 3x3, 9x9) dentro de una grilla del
    tamaño de la imagen que vamos a filtrar, y lo centra en el origen
    (esquina superior izquierda, con wrap-around) para que su FFT tenga
    la fase correcta al multiplicarla por la FFT de la imagen.

    Necesitamos que tenga las mismas dimensiones que la imagen para que resulte
    en el mismo numero de coeficientes. Por otro lado necesitamos centrar el kernel
    en el centro para mantener la intencion original.

    Utilizamos roll, ya que tratamos con un mundo 'wrappeado'.
    '''
    k_pad = np.zeros(forma)
    kh, kw = kernel.shape
    k_pad[:kh, :kw] = kernel
    k_pad = np.roll(k_pad, -(kh // 2), axis=0)
    k_pad = np.roll(k_pad, -(kw // 2), axis=1)
    return k_pad


def espectro_magnitud(imagen_o_kernel, log=True):
    '''Espectro de magnitud (centrado con fftshift) de una imagen o de un kernel ya preparado con preparar_kernel.'''
    F = np.fft.fftshift(np.fft.fft2(imagen_o_kernel))
    mag = np.abs(F)
    return np.log1p(mag) if log else mag

## Parte 1 — ¿Qué aspecto tienen nuestros kernels conocidos en frecuencia?

Antes de filtrar ninguna imagen, miremos el espectro de los kernels en sí mismos. Para eso "insertamos" cada kernel chico (3x3, 9x9, etc.) en una grilla más grande (128x128) rellena de ceros, y calculamos su FFT-2D. Esto nos deja ver con buena resolución qué frecuencias deja pasar y cuáles atenúa cada kernel.

In [ ]:
def delta(n=9):
    k = np.zeros((n, n))
    k[n // 2, n // 2] = 1.0
    return k

def gaussiana(n=9, sigma=1.5):
    eje = np.arange(n) - n // 2
    xx, yy = np.meshgrid(eje, eje)
    k = np.exp(-(xx**2 + yy**2) / (2 * sigma**2))
    return k / k.sum()

sobel_x = np.array([[-1, 0, 1],
                     [-2, 0, 2],
                     [-1, 0, 1]], dtype=float)
sobel_y = sobel_x.T

prewitt_x = np.array([[-1, 0, 1],
                       [-1, 0, 1],
                       [-1, 0, 1]], dtype=float)
prewitt_y = prewitt_x.T

roberts_x = np.array([[1, 0],
                       [0, -1]], dtype=float)
roberts_y = np.array([[0, 1],
                       [-1, 0]], dtype=float)

kernels = {
    'Delta (identidad)': delta(9),
    'Gaussiana':          gaussiana(9, 1.5),
    'Sobel x':            sobel_x,
    'Sobel y':            sobel_y,
    'Prewitt x':          prewitt_x,
    'Prewitt y':          prewitt_y,
    'Roberts x':          roberts_x,
    'Roberts y':          roberts_y,
}

In [ ]:
TAMANO_VISUALIZACION = (128, 128)

fig, axes = plt.subplots(4, 4, figsize=(14, 14))
for i, (nombre, k) in enumerate(kernels.items()):
    fila, col = i // 2, (i % 2) * 2
    axes[fila, col].imshow(k, cmap='gray')
    axes[fila, col].set_title(nombre, fontsize=10)
    axes[fila, col].axis('off')

    k_prep = preparar_kernel(k, TAMANO_VISUALIZACION)
    esp = espectro_magnitud(k_prep, log=False)  # sin log: acá nos interesa ver la magnitud "cruda"
    axes[fila, col + 1].imshow(esp, cmap='magma')
    axes[fila, col + 1].set_title(f'|FFT| de {nombre}', fontsize=10)
    axes[fila, col + 1].axis('off')

fig.tight_layout()
plt.show()

**Algunas cosas para notar:**

- El espectro de la **delta** es completamente **plano** (constante), por eso se ve "vacío" aunque sea constantemente 1. Tiene sentido: convolucionar con la delta no modifica la imagen (es el kernel identidad), así que no puede estar atenuando ninguna frecuencia.
- El espectro de la **Gaussiana** está concentrado en el centro (bajas frecuencias) y decae suavemente hacia afuera: es un filtro **pasa-bajos**. Por eso desenfoca: elimina el detalle fino (altas frecuencias) y conserva las variaciones lentas de intensidad.
- **Sobel**, **Prewitt** y **Roberts** son justo lo opuesto: tienen un valor muy bajo en el centro (frecuencia cero) y ganan energía hacia las altas frecuencias. Son filtros **pasa-altos**, y además **direccionales**: noten cómo el patrón de cada uno está orientado según el eje en el que detecta cambios (x o y).

## Parte 2 — El teorema en acción, sobre una imagen sintética

Construyamos una imagen sintética simple: la suma de dos patrones sinusoidales, uno de baja frecuencia (rayas verticales anchas) y otro de frecuencia más alta (rayas horizontales finas). Este tipo de imagen tiene un espectro muy fácil de leer: un grupo de picos, en vez de la nube difusa de una foto real.

Vamos a armar una función que muestre, para un kernel dado:

1. La **imagen** original.
2. Su **espectro**.
3. El **kernel**.
4. El **espectro del kernel**.
5. El **resultado** de filtrar (convolucionando en el dominio espacial, con `scipy.signal.convolve2d`, tal como ya lo conocen).
6. El espectro **medido**: la FFT de ese resultado filtrado.
7. El espectro **predicho por el teorema**: el producto punto a punto entre el espectro de la imagen original (2) y el espectro del kernel (4).

Si el teorema es correcto, los paneles 6 y 7 tienen que ser indistinguibles.

*Al convolucionar usamos borde periódico (`boundary='wrap') para que la comparación con la multiplicación en frecuencia sea exacta. Así evitamos diferencias en los bordes*

In [ ]:
def imagen_sintetica(n=256, f1=4, f2=20):
    eje = np.linspace(0, 1, n, endpoint=False)
    X, Y = np.meshgrid(eje, eje)
    img = np.sin(2*np.pi*f1*X) + 0.6*np.sin(2*np.pi*f2*Y)
    return (img - img.min()) / (img.max() - img.min())

sint = imagen_sintetica()

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(sint, cmap='gray'); axes[0].set_title('Imagen sintética'); axes[0].axis('off')
axes[1].imshow(espectro_magnitud(sint), cmap='magma'); axes[1].set_title('Espectro'); axes[1].axis('off')
fig.tight_layout(); plt.show()

In [ ]:
def verificar_teorema(imagen, kernel, nombre_kernel, zoom=None):
    '''
    Muestra, en orden: la imagen, su espectro, el kernel, el espectro del kernel,
    el resultado de filtrar, el espectro medido sobre ese resultado, y el espectro
    predicho por el teorema (FFT(imagen) * FFT(kernel)). Si el teorema es verdadero, los
    dos últimos paneles tienen que ser indistinguibles.
    '''
    filtrada = convolve2d(imagen, kernel, mode='same', boundary='wrap')

    F_original = np.fft.fft2(imagen)
    H = np.fft.fft2(preparar_kernel(kernel, imagen.shape))

    espectro_original  = np.fft.fftshift(F_original)
    espectro_kernel    = np.fft.fftshift(H)
    espectro_medido    = np.fft.fftshift(np.fft.fft2(filtrada))
    espectro_predicho  = np.fft.fftshift(F_original * H)

    orig_mag   = np.log1p(np.abs(espectro_original))
    kernel_mag = np.log1p(np.abs(espectro_kernel))
    med        = np.log1p(np.abs(espectro_medido))
    pred       = np.log1p(np.abs(espectro_predicho))

    if zoom:
        c0, c1 = imagen.shape[0]//2, imagen.shape[1]//2
        orig_mag   = orig_mag[c0-zoom:c0+zoom, c1-zoom:c1+zoom]
        kernel_mag = kernel_mag[c0-zoom:c0+zoom, c1-zoom:c1+zoom]
        med        = med[c0-zoom:c0+zoom, c1-zoom:c1+zoom]
        pred       = pred[c0-zoom:c0+zoom, c1-zoom:c1+zoom]

    ESCALA = dict(vmin=0, vmax=10)

    fig, axes = plt.subplots(1, 7, figsize=(21, 3.5))
    axes[0].imshow(imagen, cmap='gray');               axes[0].set_title('Imagen');              axes[0].axis('off')
    axes[1].imshow(orig_mag, cmap='magma', **ESCALA);  axes[1].set_title('Espectro (imagen)');  axes[1].axis('off')
    axes[2].imshow(kernel, cmap='gray', vmin=min(0, kernel.min()), vmax=kernel.max()); axes[2].set_title(f'Kernel\n({nombre_kernel})'); axes[2].set_xticks([]); axes[2].set_yticks([])
    axes[3].imshow(kernel_mag, cmap='magma');          axes[3].set_title('Espectro (kernel)');    axes[3].axis('off')
    axes[4].imshow(filtrada, cmap='gray');             axes[4].set_title('Resultado (filtrada)'); axes[4].axis('off')
    axes[5].imshow(med, cmap='magma', **ESCALA);       axes[5].set_title('Medido');               axes[5].axis('off')
    axes[6].imshow(pred, cmap='magma', **ESCALA);      axes[6].set_title('Predicho (teorema)');   axes[6].axis('off')
    fig.tight_layout(); plt.show()

    diferencia_max = np.abs(np.abs(espectro_medido) - np.abs(espectro_predicho)).max()
    print(f'{nombre_kernel}: diferencia máxima entre espectro medido y predicho = {diferencia_max:.2e}')

verificar_teorema(sint, gaussiana(100, 10), 'Gaussiana', zoom=40)
verificar_teorema(sint, sobel_y, 'Sobel y', zoom=40)

Los paneles "Medido" y "Predicho" son indistinguibles.

Noten también cómo se conecta todo: el "Espectro (imagen)" tiene varios picos (uno por cada componente sinusoidal). El "Espectro (kernel)" de **Sobel Y** tiene energía casi nula en la dirección horizontal y bastante en la vertical. Al multiplicarlos punto a punto, el pico que caía justo donde Sobel Y vale ~0 desaparece, por eso el "Resultado" perdió por completo las rayas anchas verticales, y el "Medido"/"Predicho" final sólo conserva un par de picos, en vez de todos los que tenía la imagen original.

## Parte 3 — Ahora con una imagen real

Repitamos el mismo experimento con una foto real (`skimage.data.camera`). El espectro va a ser mucho más rico (una foto tiene contenido en todas las frecuencias, no sólo un par de picos), pero la verificación del teorema funciona exactamente igual.

In [ ]:
foto = img_as_float(data.camera())

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(foto, cmap='gray'); axes[0].set_title('Imagen original'); axes[0].axis('off')
axes[1].imshow(espectro_magnitud(foto), cmap='magma'); axes[1].set_title('Espectro original'); axes[1].axis('off')
fig.tight_layout(); plt.show()

verificar_teorema(foto, gaussiana(15, 2.0), 'Gaussiana (desenfoque)')
verificar_teorema(foto, sobel_x, 'Sobel x (bordes verticales)')

Comparen el panel "Espectro (kernel)" con el "Resultado" y el "Medido" en cada caso: con la Gaussiana, el espectro del kernel es un blob suave concentrado en el centro, el "Resultado" se ve desenfocado, y el espectro filtrado "se apaga" hacia los bordes en la misma forma, estamos viendo al filtro pasa-bajos recortando el espectro de la foto. Con Sobel pasa lo contrario: el espectro del kernel es casi nulo en el centro, el "Resultado" muestra sólo los bordes de los objetos, y el espectro filtrado pierde toda la componente de baja frecuencia.

En ambos casos, "Medido" y "Predicho" vuelven a coincidir.

## Ejercicio 1 — Filtrar trabajando directamente en frecuencia

Hasta acá filtramos siempre en el dominio espacial (con `convolve2d`), y usamos el teorema sólo para **verificar** el resultado en frecuencia. Pero el teorema permite algo más interesante: filtrar una imagen **sin convolucionar nunca en el dominio espacial**, trabajando pura y directamente en frecuencia.

Implementen `convolucion_frecuencia(imagen, kernel)`, que:

1. Prepara el kernel al tamaño de la imagen (con `preparar_kernel`, ya definida más arriba).
2. Calcula la FFT-2D de la imagen.
3. Calcula la FFT-2D del kernel preparado.
4. Multiplica ambas FFTs punto a punto.
5. Aplica la FFT-2D inversa y devuelve la parte real (los residuos imaginarios que puedan quedar son error numérico de punto flotante).

Después de implementarla, corran la celda de tests para chequear que el resultado coincide con `convolve2d(imagen, kernel, mode='same', boundary='wrap')`.

In [ ]:
def convolucion_frecuencia(imagen, kernel):
    '''
    Filtra 'imagen' con 'kernel' trabajando en el dominio de la frecuencia.

    Parámetros
    ----------
    imagen : ndarray 2D
    kernel : ndarray 2D, más chico que 'imagen'

    Devuelve
    --------
    ndarray 2D del mismo tamaño que 'imagen', con el filtro aplicado.
    '''
    # TODO: completar
    # Paso 1: preparar el kernel al tamaño de la imagen
    # Paso 2 y 3: FFT-2D de la imagen y del kernel preparado
    # Paso 4: multiplicar punto a punto
    # Paso 5: FFT-2D inversa, quedarse con la parte real
    raise NotImplementedError("Completar convolucion_frecuencia")

In [ ]:
def _test_convolucion_frecuencia():
    rng = np.random.default_rng(0)
    imagen_test = rng.random((64, 64))
    casos = {
        'Delta':     delta(5),
        'Gaussiana': gaussiana(7, 1.2),
        'Sobel x':   sobel_x,
        'Prewitt y': prewitt_y,
    }
    ok_total = True
    for nombre, k in casos.items():
        esperado = convolve2d(imagen_test, k, mode='same', boundary='wrap')
        try:
            obtenido = convolucion_frecuencia(imagen_test, k)
        except NotImplementedError:
            print(f'❌ {nombre}: la función todavía no está implementada.')
            ok_total = False
            continue
        if np.allclose(obtenido, esperado, atol=1e-6):
            print(f'✅ {nombre}: correcto (diferencia máxima {np.abs(obtenido-esperado).max():.2e})')
        else:
            print(f'❌ {nombre}: no coincide con la referencia (diferencia máxima {np.abs(obtenido-esperado).max():.2e})')
            ok_total = False
    print('\n🎉 ¡Todos los tests pasaron!' if ok_total else '\nRevisen la implementación, todavía hay tests que fallan.')
    return ok_total

_ = _test_convolucion_frecuencia()

## Parte 4 — Usando el teorema al revés: deconvolución

Supongamos que tenemos una imagen $g$ que sabemos que es el resultado de convolucionar una imagen original $f$ (que no tenemos) con un kernel $h$ que sí conocemos:

$$ g = f * h $$

Por el teorema de convolución:

$$ \mathcal{F}\{g\} = \mathcal{F}\{f\} \cdot \mathcal{F}\{h\} $$

Como conocemos $h$ (y por lo tanto $\mathcal{F}\{h\}$), y tenemos $g$ (y por lo tanto $\mathcal{F}\{g\}$), podemos **despejar** $\mathcal{F}\{f\}$:

$$ \mathcal{F}\{f\} = \frac{\mathcal{F}\{g\}}{\mathcal{F}\{h\}} \quad \Longrightarrow \quad f = \mathcal{F}^{-1}\left\{\frac{\mathcal{F}\{g\}}{\mathcal{F}\{h\}}\right\} $$

Esto es **deconvolución**: recuperar la imagen original a partir de la imagen degradada y el kernel conocido, dividiendo en frecuencia en vez de multiplicar.

Generemos primero una imagen "degradada" a propósito, usando la función que implementaron en el Ejercicio 1: convolucionamos la foto con un kernel Gaussiano conocido. *(Esta celda usa `convolucion_frecuencia`, así que necesitan haber resuelto el Ejercicio 1 para que funcione.)*

In [ ]:
kernel_conocido = gaussiana(9, 2.0)
foto_borrosa = convolucion_frecuencia(foto, kernel_conocido)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(foto, cmap='gray'); axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(foto_borrosa, cmap='gray'); axes[1].set_title('Degradada (convolucionada con Gaussiana)'); axes[1].axis('off')
fig.tight_layout(); plt.show()

## Ejercicio 2 — Implementar la deconvolución

Implementen `deconvolucion_frecuencia(imagen, kernel)`, que revierte el proceso: recibe la imagen degradada y el kernel conocido, y devuelve la estimación de la imagen original, usando la fórmula de más arriba (dividir en frecuencia en vez de multiplicar).

Va a ser muy parecida a `convolucion_frecuencia`, con una diferencia clave.

In [ ]:
def deconvolucion_frecuencia(imagen, kernel):
    '''
    Estima la imagen original a partir de 'imagen' (degradada) y 'kernel' (conocido),
    asumiendo que imagen = convolucion_frecuencia(original, kernel), sin ruido.

    Parámetros
    ----------
    imagen : ndarray 2D (degradada)
    kernel : ndarray 2D, el kernel conocido que generó la degradación

    Devuelve
    --------
    ndarray 2D: estimación de la imagen original.
    '''
    # TODO: completar (muy similar a convolucion_frecuencia, pero dividiendo en vez de multiplicar)
    raise NotImplementedError("Completar deconvolucion_frecuencia")

In [ ]:
def _test_deconvolucion_frecuencia():
    kernel_prueba = gaussiana(9, 2.0)
    degradada = convolucion_frecuencia(foto, kernel_prueba)
    try:
        recuperada = deconvolucion_frecuencia(degradada, kernel_prueba)
    except NotImplementedError:
        print('❌ La función todavía no está implementada.')
        return False

    diferencia = np.abs(foto - recuperada).max()
    ok = np.allclose(foto, recuperada, atol=1e-4)

    espectro_foto       = np.fft.fftshift(np.fft.fft2(foto))
    espectro_kernel     = np.fft.fftshift(np.fft.fft2(preparar_kernel(kernel_prueba, foto.shape)))
    espectro_degradada  = np.fft.fftshift(np.fft.fft2(degradada))
    espectro_recuperada = np.fft.fftshift(np.fft.fft2(recuperada))

    mag_foto       = np.log1p(np.abs(espectro_foto))
    mag_kernel     = np.log1p(np.abs(espectro_kernel))
    mag_degradada  = np.log1p(np.abs(espectro_degradada))
    mag_recuperada = np.log1p(np.abs(espectro_recuperada))

    ESCALA = dict(vmin=0, vmax=10)

    fig, axes = plt.subplots(1, 8, figsize=(24, 3.3))
    axes[0].imshow(foto, cmap='gray');                      axes[0].set_title('Original');            axes[0].axis('off')
    axes[1].imshow(mag_foto, cmap='magma', **ESCALA);       axes[1].set_title('Espectro');            axes[1].axis('off')
    axes[2].imshow(kernel_prueba, cmap='gray', vmin=min(0, kernel_prueba.min()), vmax=kernel_prueba.max()); axes[2].set_title('Kernel\n(degradante)'); axes[2].set_xticks([]); axes[2].set_yticks([])
    axes[3].imshow(mag_kernel, cmap='magma');               axes[3].set_title('Espectro (kernel)');   axes[3].axis('off')
    axes[4].imshow(degradada, cmap='gray');                 axes[4].set_title('Degradada');           axes[4].axis('off')
    axes[5].imshow(mag_degradada, cmap='magma', **ESCALA);  axes[5].set_title('Espectro');            axes[5].axis('off')
    axes[6].imshow(recuperada, cmap='gray');                axes[6].set_title('Recuperada');          axes[6].axis('off')
    axes[7].imshow(mag_recuperada, cmap='magma', **ESCALA); axes[7].set_title('Espectro');            axes[7].axis('off')
    fig.tight_layout(); plt.show()

    if ok:
        print(f'✅ Correcto: la imagen recuperada coincide con la original (diferencia máxima {diferencia:.2e}).')
    else:
        print(f'❌ La imagen recuperada no coincide con la original (diferencia máxima {diferencia:.2e}).')
    return ok

_ = _test_deconvolucion_frecuencia()

## Parte 5 — Cuando el teorema mismo te avisa que algo no va a andar

En el ejercicio anterior la deconvolución funcionó casi a la perfección (sin ruido de por medio). Pero la fórmula

$$ \mathcal{F}\{f\} = \frac{\mathcal{F}\{g\}}{\mathcal{F}\{h\}} $$

tiene un problema evidente: **si $\mathcal{F}\{h\}$ vale exactamente cero en alguna frecuencia, ahí estamos dividiendo por cero. Si $h$ elimina por completo cierta frecuencia al convolucionar, esa información **ya no está** en $g$, y no hay forma de recuperarla dividiendo.

Veamos un ejemplo concreto: un kernel "caja" (promediado uniforme) de $8\times8$ sobre nuestra foto de $512\times512$. La FFT de este kernel tiene ceros exactos en varias frecuencias.

In [ ]:
def caja(n):
    k = np.ones((n, n))
    pad = 3  # agregamos padding para no ver una caja blanca en el display
    k_padded = np.pad(k, pad, mode='constant', constant_values=0)
    return k_padded / k_padded.sum()

kernel_caja = caja(8)  # 512 / 8 = 64 exacto -> hay ceros exactos en la FFT

H_caja = np.fft.fft2(preparar_kernel(kernel_caja, foto.shape))
print('Valor mínimo de |H(u,v)| para el kernel caja de 8x8:', np.abs(H_caja).min())
print('Cantidad de frecuencias con |H(u,v)| prácticamente cero:', np.sum(np.abs(H_caja) < 1e-10))

degradada_caja = convolucion_frecuencia(foto, kernel_caja)

with np.errstate(divide='ignore', invalid='ignore'):
    recuperada_caja = deconvolucion_frecuencia(degradada_caja, kernel_caja)

fraccion_rota = (~np.isfinite(recuperada_caja)).mean()
print(f'Fracción de píxeles que dieron NaN/Inf al "recuperar": {fraccion_rota:.0%}')

recuperada_caja_vis = np.nan_to_num(recuperada_caja, nan=0, posinf=0, neginf=0)

mag_foto            = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(foto))))
mag_kernel_caja      = np.log1p(np.abs(np.fft.fftshift(H_caja)))
mag_degradada_caja   = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(degradada_caja))))
mag_recuperada_caja  = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(recuperada_caja_vis))))

ESCALA = dict(vmin=0, vmax=10)

fig, axes = plt.subplots(1, 8, figsize=(24, 3.3))
axes[0].imshow(foto, cmap='gray');                           axes[0].set_title('Original');             axes[0].axis('off')
axes[1].imshow(mag_foto, cmap='magma', **ESCALA);            axes[1].set_title('Espectro');             axes[1].axis('off')
axes[2].imshow(kernel_caja, cmap='gray', vmin=0, vmax=.2); axes[2].set_title('Kernel\n(caja 8x8)');   axes[2].set_xticks([]); axes[2].set_yticks([])
axes[3].imshow(mag_kernel_caja, cmap='magma');               axes[3].set_title('Espectro (kernel)');    axes[3].axis('off')
axes[4].imshow(degradada_caja, cmap='gray');                 axes[4].set_title('Degradada');            axes[4].axis('off')
axes[5].imshow(mag_degradada_caja, cmap='magma', **ESCALA); axes[5].set_title('Espectro');             axes[5].axis('off')
axes[6].imshow(recuperada_caja_vis, cmap='gray');            axes[6].set_title('"Recuperada"\n(rota)'); axes[6].axis('off')
axes[7].imshow(mag_recuperada_caja, cmap='magma', **ESCALA);axes[7].set_title('Espectro');             axes[7].axis('off')
fig.tight_layout(); plt.show()

## Parte 6 — Un arreglo simple: una constante en el denominador

Una forma sencilla de evitar la división por cero es sumar una constante chica $K$ al denominador:

$$ \mathcal{F}\{f\} \approx \frac{\mathcal{F}\{g\}}{\mathcal{F}\{h\} + K} $$

Con $K > 0$, el denominador ya nunca es exactamente cero. Esto **no recupera** la información que el kernel eliminó, pero evita que un único cero exacto rompa toda la imagen con NaN/Inf, y a cambio deja algún artefacto donde antes había un cero (o casi-cero).

In [ ]:
def deconvolucion_regularizada(imagen, kernel, K=1e-2):
    k_prep = preparar_kernel(kernel, imagen.shape)
    G = np.fft.fft2(imagen)
    H = np.fft.fft2(k_prep)
    return np.fft.ifft2(G / (H + K)).real

recuperada_caja_regularizada = deconvolucion_regularizada(degradada_caja, kernel_caja, K=1e-2)

print('¿Tiene NaN/Inf?', np.any(~np.isfinite(recuperada_caja_regularizada)))

# Reutilizamos foto, kernel_caja, degradada_caja y sus espectros ya calculados arriba.
mag_recuperada_reg = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(recuperada_caja_regularizada))))

fig, axes = plt.subplots(1, 8, figsize=(24, 3.3))
axes[0].imshow(foto, cmap='gray');                          axes[0].set_title('Original');            axes[0].axis('off')
axes[1].imshow(mag_foto, cmap='magma', **ESCALA);           axes[1].set_title('Espectro');            axes[1].axis('off')
axes[2].imshow(kernel_caja, cmap='gray', vmin=0, vmax=1); axes[2].set_title('Kernel\n(caja 8x8)');  axes[2].set_xticks([]); axes[2].set_yticks([])
axes[3].imshow(mag_kernel_caja, cmap='magma');              axes[3].set_title('Espectro (kernel)');   axes[3].axis('off')
axes[4].imshow(degradada_caja, cmap='gray');                axes[4].set_title('Degradada');           axes[4].axis('off')
axes[5].imshow(mag_degradada_caja, cmap='magma', **ESCALA);axes[5].set_title('Espectro');            axes[5].axis('off')
axes[6].imshow(recuperada_caja_regularizada, cmap='gray');  axes[6].set_title('Recuperada\n(con K)'); axes[6].axis('off')
axes[7].imshow(mag_recuperada_reg, cmap='magma', **ESCALA);axes[7].set_title('Espectro');            axes[7].axis('off')
fig.tight_layout(); plt.show()

## Ejercicio 3 — El telescopio averiado

Sos la ingeniere jefe de una nave espacial. El telescopio principal viene fallando: cada vez que se toma un **lote** de fotos, el sistema óptico le agrega una distorsión nueva a todas las imágenes de ese lote. El equipo de óptica determinó que la distorsión es, matemáticamente, una convolución entre cada imagen y un kernel desconocido.

No se puede reparar el telescopio hasta llegar a la próxima estación espacial. Como ingeniere jefe, tu trabajo es diseñar un parche temporal para recuperar la mayor cantidad de contenido original posible de cada lote, **sin conocer el kernel de antemano.**

Durante una reuninon se te ocurre una idea: podés poner rápidamente una imagen conocida delante del telescopio para que quede incluida en el mismo lote. Como el kernel es el mismo para *todas* las fotos de un lote, esta imagen sera afectada por la misma transformacion.

Les dejamos ya armada la función `telescopio(lote_de_fotos)`: recibe una lista de imágenes (todas del mismo tamaño), les aplica una convolución con un kernel aleatorio, nuevo en cada llamada, y devuelve la lista resultante. No hay forma de ver ese kernel directamente.

*(La celda de abajo descarga imágenes reales del [dataset "NASA Image of the Day" de Kaggle](https://www.kaggle.com/datasets/auhide/nasa-image-of-the-day-dataset))*

In [ ]:
!pip install -q kagglehub

import glob
import os
import random
import kagglehub
from skimage import io, transform, color

ruta_dataset = kagglehub.dataset_download("auhide/nasa-image-of-the-day-dataset")
print("Dataset descargado en:", ruta_dataset)

extensiones = ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG')
archivos = []
for ext in extensiones:
    archivos.extend(glob.glob(os.path.join(ruta_dataset, '**', ext), recursive=True))

print(f"Se encontraron {len(archivos)} imágenes.")
if len(archivos) < 3:
    raise RuntimeError(
        f"Se esperaban al menos 3 imágenes y sólo se encontraron {len(archivos)} en '{ruta_dataset}'. "
        "Puede que la estructura de carpetas del dataset haya cambiado, revisen "
        "'ruta_dataset' a mano (por ejemplo con os.walk) para ajustar las extensiones o la búsqueda."
    )


def cargar_como_gris_512(ruta_archivo):
    '''
    Carga una imagen de disco, la pasa a escala de grises, recorta el
    cuadrado central más grande posible (para no deformar la imagen al
    escalar) y la redimensiona a 512x512 en float [0,1].
    '''
    img = io.imread(ruta_archivo)
    if img.ndim == 3:
        img = color.rgb2gray(img[..., :3])   # descartamos canal alfa si lo hay
    alto, ancho = img.shape
    lado = min(alto, ancho)
    y0, x0 = (alto - lado) // 2, (ancho - lado) // 2
    img = img[y0:y0+lado, x0:x0+lado]
    return transform.resize(img, (512, 512), anti_aliasing=True).astype(float)

In [ ]:
random.seed(125)
elegidas = random.sample(archivos, 3)
nombres_apod = [os.path.splitext(os.path.basename(a))[0] for a in elegidas]
img_1, img_2, img_3 = [cargar_como_gris_512(a) for a in elegidas]

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, img, nombre in zip(axes, [img_1, img_2, img_3], nombres_apod):
    ax.imshow(img, cmap='gray'); ax.set_title(nombre, fontsize=9); ax.axis('off')
fig.tight_layout(); plt.show()

In [ ]:
def kernel_aleatorio(n=21, seed=None):
    '''Un kernel con una forma entre dos familias posibles, con parámetros al azar.'''
    rng = np.random.default_rng(seed)
    eje = np.linspace(-2, 2, n)
    X, Y = np.meshgrid(eje, eje)
    tipo = rng.choice(['gaussiana', 'apertura'])
    if tipo == 'gaussiana':
        sx = rng.uniform(0.3, 2.1)
        sy = rng.uniform(0.3, 2.1)
        k = np.exp(-(X**2/(2*sx**2) + Y**2/(2*sy**2)))
    else:
        a = rng.uniform(0.6, 2)
        b = rng.uniform(0.6, 2)
        valor = (X/a)**2 + np.log((Y/b)**2 + 0.05)   # la curva x²+log(y²+ε)=1
        mascara = (valor <= 1).astype(float)
        k = mascara * np.exp(-(X**2 + Y**2) / (2*1.8**2))  # afinamos el borde
    return k / k.sum()


def telescopio(lote_de_fotos, seed=None):
    '''
    La caja negra del telescopio averiado. Recibe una lista de imágenes
    (todas del mismo tamaño) y devuelve una nueva lista, cada una
    convolucionada por UN ÚNICO kernel aleatorio, no tienen forma de conocer de antemano.

    'seed' es opcional: sin especificarlo, cada llamada usa un kernel
    distinto (como en la nave real). Fijarlo sirve para debuggear o para
    poder mostrar el kernel usado en una corrida puntual.
    '''
    kernel_secreto = kernel_aleatorio(seed=seed)
    return [convolucion_frecuencia(imagen, kernel_secreto) for imagen in lote_de_fotos]


def deconvolucion_wiener(imagen, kernel, K=1e-3):
    '''
    Como la deconvolución regularizada de la Parte 6, pero con |H|^2 en vez
    de H en el denominador:

        F(f) ≈ conj(H) · F(g) / (|H|^2 + K)

    H puede dar cualquier número complejo, y "H + K" puede seguir siendo chico
    si la parte real de H resulta negativa y cercana a -K.
    En cambio |H|^2 siempre es real y no negativo, así que |H|^2 + K nunca
    se acerca a cero mientras K > 0: funciona sin importar qué kernel toque.
    '''
    k_prep = preparar_kernel(kernel, imagen.shape)
    G = np.fft.fft2(imagen)
    H = np.fft.fft2(k_prep)
    F = np.conj(H) * G / (np.abs(H)**2 + K)
    return np.fft.ifft2(F).real

**La consigna:** implementen `reparar_lote(fotos)`, que:

1. Recibe `fotos`, una lista de imágenes originales.
2. Arma un lote para enviar al telescopio, pueden agregar imágenes extra al lote, pero **las imágenes originales tienen que viajar sin modificar** (no vale procesarlas antes de mandarlas).
3. Llama a `telescopio(...)` con ese lote.
4. A partir del resultado, recupera el kernel que usó el telescopio en esa llamada.
5. Devuelve una lista con las versiones recuperadas de cada imagen original.

*Ayuda:* ¿qué imagen nos podria ayudar para conseguir el kernel?

Usen `deconvolucion_wiener` (ya definida arriba) para la reconstrucción.

In [ ]:
def delta_en_centro(forma):
    '''Imagen del tamaño 'forma', toda cero salvo un 1 justo en el centro.'''
    d = np.zeros(forma)
    d[forma[0] // 2, forma[1] // 2] = 1.0
    return d


def reparar_lote(fotos, seed=None):
    '''
    Recibe una lista de imágenes 'fotos' (todas del mismo tamaño), las hace
    pasar por 'telescopio' junto con alguna imagen adicional que revele el
    kernel usado, y devuelve una lista del mismo largo con las versiones
    recuperadas de cada imagen original.

    'seed' es opcional y hay que pasárselo a 'telescopio' tal cual, así se
    puede reproducir una corrida puntual.
    '''
    # TODO: completar
    raise NotImplementedError("Completar reparar_lote")

In [ ]:
import random

def _test_reparar_lote(n_pruebas=8):
    fotos = [img_1, img_2, img_3]
    nombres = nombres_apod
    ok_total = True
    peor_mae = 0.0

    for prueba in range(n_pruebas):
        try:
            recuperadas = reparar_lote(fotos)   # sin seed: un kernel nuevo en cada prueba
        except NotImplementedError:
            print('❌ La función todavía no está implementada.')
            return False

        if len(recuperadas) != len(fotos):
            print(f'❌ Prueba {prueba+1}: se esperaban {len(fotos)} imágenes recuperadas, llegaron {len(recuperadas)}.')
            ok_total = False
            continue

        maes = [np.abs(orig - rec).mean() for orig, rec in zip(fotos, recuperadas)]
        peor_mae = max(peor_mae, max(maes))
        estado = '✅' if max(maes) < 0.08 else '❌'
        if estado == '❌':
            ok_total = False
        print(f'{estado} Prueba {prueba+1}: MAE máximo entre las 3 fotos = {max(maes):.4f}')

    if not ok_total:
        print(f'\nRevisen la implementación, peor MAE observado: {peor_mae:.4f}')
        return False

    print('\n🎉 ¡Todas las pruebas pasaron! (el kernel fue distinto en cada una)')

    # Diagnóstico completo con semilla fija, para poder mostrar también el
    # kernel real y la imagen tal como salió del telescopio.
    SEMILLA_DEMO = random.randint(1, 100)
    kernel_real  = kernel_aleatorio(seed=SEMILLA_DEMO)
    modificadas  = telescopio(fotos, seed=SEMILLA_DEMO)
    recuperadas  = reparar_lote(fotos, seed=SEMILLA_DEMO)

    ESCALA = dict(vmin=0, vmax=10)
    fig, axes = plt.subplots(len(fotos), 8, figsize=(24, 3.3*len(fotos)))
    for i, (orig, mod, rec, nombre) in enumerate(zip(fotos, modificadas, recuperadas, nombres)):
        esp_orig   = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(orig))))
        esp_kernel = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(preparar_kernel(kernel_real, orig.shape)))))
        esp_mod    = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(mod))))
        esp_rec    = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(rec))))

        axes[i,0].imshow(orig, cmap='gray');                          axes[i,0].set_title(nombre);            axes[i,0].axis('off')
        axes[i,1].imshow(esp_orig, cmap='magma', **ESCALA);           axes[i,1].set_title('Espectro');        axes[i,1].axis('off')
        axes[i,2].imshow(kernel_real, cmap='gray', vmin=min(0, kernel_real.min()), vmax=kernel_real.max())
        axes[i,2].set_title('Kernel'); axes[i,2].set_xticks([]); axes[i,2].set_yticks([])
        axes[i,3].imshow(esp_kernel, cmap='magma');                   axes[i,3].set_title('Espectro (kernel)'); axes[i,3].axis('off')
        axes[i,4].imshow(mod, cmap='gray');                           axes[i,4].set_title('Modificada');      axes[i,4].axis('off')
        axes[i,5].imshow(esp_mod, cmap='magma', **ESCALA);            axes[i,5].set_title('Espectro');        axes[i,5].axis('off')
        axes[i,6].imshow(rec, cmap='gray');                           axes[i,6].set_title('Recuperada');      axes[i,6].axis('off')
        axes[i,7].imshow(esp_rec, cmap='magma', **ESCALA);            axes[i,7].set_title('Espectro');        axes[i,7].axis('off')
    fig.tight_layout(); plt.show()

    return True

_ = _test_reparar_lote()

# Sección 2 — Ruido periódico: detectarlo y filtrarlo

Hay un tipo de ruido muy distinto al que vimos hasta ahora: el **ruido periódico**. Aparece, por ejemplo, en escáneres viejos, interferencia eléctrica, o sensores con algún patrón de lectura regular. En vez de ser aleatorio píxel a píxel, se repite con una periodicidad fija en el espacio (rayas, tramas, patrones).

Vamos a ver qué aspecto tiene este ruido en el dominio de la frecuencia, cómo aislarlo y eliminarlo cuando sabemos dónde está, y después cómo **detectarlo automáticamente** cuando no lo sabemos.

## ¿Qué aspecto tiene en frecuencia?

Un patrón periódico puro (una sola frecuencia espacial) es, ni más ni menos, una sinusoide. Y ya sabemos qué hace el teorema con eso: una sinusoide pura tiene **toda su energía concentrada en un conjunto de frecuencias**, la frecuencia y su conjugada (para que la señal resultante sea real). En la imagen se ve como un patrón repetitivo; en el espectro, como un par de puntos brillantes y simétricos respecto al centro.

In [ ]:
def ruido_periodico(forma, fu, fv, amplitud=0.15, fase=0.0):
    '''Una componente sinusoidal pura, con frecuencia (fu,fv) en ciclos por imagen.'''
    x = np.arange(forma[1]); y = np.arange(forma[0])
    X, Y = np.meshgrid(x, y)
    return amplitud * np.sin(2*np.pi*(fu*X/forma[1] + fv*Y/forma[0]) + fase)

ESCALA_RUIDO = dict(vmin=0, vmax=9)
ESCALA_RESIDUAL = dict(vmin=0, vmax=5)

ruido_horizontal = ruido_periodico(foto.shape, fu=0, fv=25, amplitud=0.15)
ruido_diagonal   = ruido_periodico(foto.shape, fu=25, fv=25, amplitud=0.12)

espectro_original = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(foto))))

fig, axes = plt.subplots(3, 3, figsize=(13, 12))
filas = ['Imagen', 'Espectro', 'Residual\n(espectro - espectro original)']
casos = [('Original', np.zeros_like(foto)), ('+ rayas horizontales', ruido_horizontal), ('+ patrón diagonal', ruido_diagonal)]
for col, (nombre, ruido) in enumerate(casos):
    imagen = foto + ruido
    espectro = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(imagen))))
    residual = espectro - espectro_original   # la FFT es lineal: esto aisla justo lo que agregó el ruido

    axes[0, col].imshow(imagen, cmap='gray', vmin=0, vmax=1); axes[0, col].set_title(nombre); axes[0, col].axis('off')
    axes[1, col].imshow(espectro, cmap='magma', **ESCALA_RUIDO); axes[1, col].axis('off')
    axes[2, col].imshow(residual, cmap='magma', **ESCALA_RESIDUAL); axes[2, col].axis('off')

for fila, etiqueta in enumerate(filas):
    axes[fila, 0].text(-0.08, 0.5, etiqueta, transform=axes[fila, 0].transAxes,
                        rotation=90, va='center', ha='right', fontsize=10)

fig.tight_layout(); plt.show()

Miren la fila del medio: en la imagen original el espectro es la nube difusa de siempre. En las otras dos aparece, además, un par de puntitos brillantes y simétricos respecto al centro. Cuantas más rayas por unidad de longitud (mayor frecuencia), más lejos del centro aparece el punto.

La fila de abajo lo deja todavía más claro. Como la FFT es lineal, el espectro de "imagen + ruido" es la suma de sus espectros — así que restarle el espectro original a cada uno aísla, casi a la perfección, nada más que lo que agregó el ruido: negro en todos lados salvo esos dos puntitos.

## Aislar una frecuencia puntual

Si sabemos exactamente en qué frecuencia vive el ruido (como acá, porque lo pusimos nosotros), podemos quedarnos *sólo* con esas frecuencias (todo el resto en cero) y reconstruir, para confirmar que ahí vive exactamente el patrón que sospechamos. A esto lo llamamos **aislar**.

In [ ]:
def mascara_desde_frecuencias(forma, pares_fu_fv):
    '''Máscara booleana: True en las posiciones (y sus conjugadas) de una lista de frecuencias (fu,fv) conocidas.'''
    c0, c1 = forma[0] // 2, forma[1] // 2
    mascara = np.zeros(forma, dtype=bool)
    for fu, fv in pares_fu_fv:
        mascara[c0+fv, c1+fu] = True
        mascara[c0-fv, c1-fu] = True
    return mascara


def aislar_frecuencias(imagen, mascara):
    '''Reconstruye SÓLO lo que vive en las frecuencias marcadas por la máscara.'''
    F = np.fft.fftshift(np.fft.fft2(imagen))
    F_aislado = np.zeros_like(F)
    F_aislado[mascara] = F[mascara]
    return np.fft.ifft2(np.fft.ifftshift(F_aislado)).real


imagen_con_ruido = foto + ruido_diagonal
mascara_conocida = mascara_desde_frecuencias(foto.shape, [(25, 25)])
solo_ruido = aislar_frecuencias(imagen_con_ruido, mascara_conocida)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(imagen_con_ruido, cmap='gray', vmin=0, vmax=1); axes[0].set_title('Con ruido'); axes[0].axis('off')
axes[1].imshow(solo_ruido, cmap='gray'); axes[1].set_title('Aislado (sólo esa frecuencia)'); axes[1].axis('off')
fig.tight_layout(); plt.show()

El panel "Aislado" confirma la sospecha: esas dos frecuencias, y nada más, son exactamente el patrón diagonal que agregamos, una sinusoide perfecta.

*(Acá no usamos `np.clip` para mantener los valores en $[0,1]$ como en otras partes del notebook, si recortáramos los valores fuera de rango, esa no linealidad mancharía el espectro con armónicos que no estaban en el ruido original. Usamos `vmin=0, vmax=1` sólo para mostrar la imagen, sin tocar los datos.)*

## Ejercicio 4 — Eliminar una frecuencia puntual

`aislar_frecuencias` se queda con lo que vive *adentro* de la máscara. Ahora implementen la operación complementaria: `eliminar_frecuencias(imagen, mascara)`, que se queda con todo lo que vive *afuera* de la máscara, es decir, la reconstrucción sin esas frecuencias.

In [ ]:
def eliminar_frecuencias(imagen, mascara):
    '''
    Reconstruye todo MENOS lo que vive en las frecuencias marcadas por la
    máscara.
    '''
    # TODO: completar
    raise NotImplementedError("Completar eliminar_frecuencias")

In [ ]:
def _test_eliminar_frecuencias(n_pruebas=4):
    ok_total = True
    for prueba in range(n_pruebas):
        rng = np.random.default_rng(prueba)
        mascara = rng.random(foto.shape) < 0.0005   # máscara al azar, sin nada especial
        try:
            eliminado = eliminar_frecuencias(foto, mascara)
        except NotImplementedError:
            print('❌ La función todavía no está implementada.')
            return False

        # aislar_frecuencias ya está dada y es correcta: si "eliminar" también lo
        # es, aislado + eliminado tiene que reconstruir la imagen exacta, porque
        # entre las dos cubren TODAS las frecuencias sin superponerse.
        aislado = aislar_frecuencias(foto, mascara)
        diferencia = np.abs((aislado + eliminado) - foto).max()
        estado = '✅' if diferencia < 1e-6 else '❌'
        if estado == '❌':
            ok_total = False
        print(f'{estado} Prueba {prueba+1}: |aislado + eliminado - original| máximo = {diferencia:.2e}')

    print('\n🎉 ¡Todas las pruebas pasaron!' if ok_total else '\nRevisen la implementación.')
    return ok_total

_ = _test_eliminar_frecuencias()

## Detectarlo sin saber dónde está

Hasta acá hicimos trampa: sabíamos exactamente en qué frecuencia buscar porque nosotros pusimos el ruido ahí. En la vida real no vamos a tener esa información. Necesitamos una forma de **detectar automáticamente** esos puntos brillantes y aislados.

La idea es simple: un punto de ruido periódico se destaca *muchísimo* sobre su entorno inmediato, es un pico aislado, no una zona que crece gradualmente como el resto del espectro. Alcanza con comparar la magnitud de cada punto contra la mediana de sus vecinos cercanos: si es varias veces más grande, es sospechoso. Excluimos además un disco chico alrededor del centro, porque ahí vive legítimamente el grueso del contenido de cualquier imagen natural (no queremos "detectar" la imagen misma como ruido). Les damos armada `mascara_circular` para esa parte.

In [ ]:
def mascara_circular(forma, radio):
    '''Máscara booleana: True en los puntos que están a distancia <= radio del centro de 'forma'.'''
    alto, ancho = forma
    c0, c1 = alto // 2, ancho // 2
    Y, X = np.ogrid[:alto, :ancho]
    return (Y - c0)**2 + (X - c1)**2 <= radio**2

## Ejercicio 5 — Detectar los picos automáticamente

Implementen `detectar_picos_periodicos(magnitud, radio_exclusion=8, tamano_ventana=9, factor_umbral=6)`, que devuelva una máscara booleana marcando los píxeles del espectro de magnitud que:

1. Están a más de `radio_exclusion` píxeles del centro (usen `mascara_circular` para esto).
2. Superan en más de `factor_umbral` veces la mediana de una ventana de `tamano_ventana` × `tamano_ventana` a su alrededor (`scipy.ndimage.median_filter` calcula justo eso).

In [ ]:
from scipy.ndimage import median_filter

def detectar_picos_periodicos(magnitud, radio_exclusion=8, tamano_ventana=9, factor_umbral=6):
    '''
    Devuelve una máscara booleana marcando los píxeles del espectro de
    magnitud que se destacan mucho sobre la mediana de su entorno,
    candidatos a ser picos de ruido periódico, excluyendo un disco de
    radio 'radio_exclusion' alrededor de la frecuencia cero.
    '''
    # TODO: completar
    raise NotImplementedError("Completar detectar_picos_periodicos")

In [ ]:
def _test_detectar_picos_periodicos(n_pruebas=5):
    ok_total = True
    x = np.arange(foto.shape[1]); y = np.arange(foto.shape[0])
    X, Y = np.meshgrid(x, y)
    c0, c1 = foto.shape[0] // 2, foto.shape[1] // 2

    for prueba in range(n_pruebas):
        rng = np.random.default_rng(prueba)
        n_componentes = rng.integers(1, 3)
        ruido = np.zeros(foto.shape)
        picos_verdaderos = []
        for _ in range(n_componentes):
            radio = rng.uniform(30, 100)
            angulo = rng.uniform(0, 2*np.pi)
            fu = int(round(radio * np.cos(angulo)))
            fv = int(round(radio * np.sin(angulo)))
            amplitud = rng.uniform(0.08, 0.2)
            ruido += amplitud * np.sin(2*np.pi*(fu*X/foto.shape[1] + fv*Y/foto.shape[0]) + rng.uniform(0, 2*np.pi))
            picos_verdaderos.append((fu, fv))

        magnitud = np.abs(np.fft.fftshift(np.fft.fft2(foto + ruido)))
        try:
            mascara = detectar_picos_periodicos(magnitud)
        except NotImplementedError:
            print('❌ La función todavía no está implementada.')
            return False

        todos_encontrados = all(
            mascara[c0+fv, c1+fu] and mascara[c0-fv, c1-fu]
            for fu, fv in picos_verdaderos
        )
        estado = '✅' if todos_encontrados else '❌'
        if not todos_encontrados:
            ok_total = False
        print(f'{estado} Prueba {prueba+1}: {len(picos_verdaderos)} frecuencia(s) inyectada(s), ¿todas detectadas? {todos_encontrados}')

    print('\n🎉 ¡Todas las pruebas pasaron!' if ok_total else '\nRevisen la implementación.')
    return ok_total

_ = _test_detectar_picos_periodicos()

## Ejercicio 6 — Filtrar ruido periódico desconocido

Les dejamos armada `agregar_ruido_periodico(imagen, seed=None)`: le agrega a una imagen entre 1 y 3 componentes de ruido periódico, con frecuencias, amplitudes y fases al azar, ustedes no ven cuáles.

**La consigna:** implementen `limpiar_ruido_periodico(imagen)`, que reciba una imagen con ruido periódico desconocido y devuelva una estimación de la imagen limpia, apoyándose en `detectar_picos_periodicos` y `eliminar_frecuencias` que ya implementaron.

In [ ]:
def agregar_ruido_periodico(imagen, seed=None):
    '''Agrega entre 1 y 3 componentes de ruido periódico sinusoidal, con
    frecuencias, amplitudes y fases al azar. Devuelve la imagen contaminada.'''
    rng = np.random.default_rng(seed)
    n_componentes = rng.integers(1, 4)
    x = np.arange(imagen.shape[1]); y = np.arange(imagen.shape[0])
    X, Y = np.meshgrid(x, y)
    ruido = np.zeros(imagen.shape)
    for _ in range(n_componentes):
        radio = rng.uniform(20, 120)
        angulo = rng.uniform(0, 2*np.pi)
        fu = int(round(radio * np.cos(angulo)))
        fv = int(round(radio * np.sin(angulo)))
        amplitud = rng.uniform(0.05, 0.2)
        fase = rng.uniform(0, 2*np.pi)
        ruido += amplitud * np.sin(2*np.pi*(fu*X/imagen.shape[1] + fv*Y/imagen.shape[0]) + fase)
    return imagen + ruido


def limpiar_ruido_periodico(imagen):
    '''
    Recibe una imagen con ruido periódico desconocido y devuelve una
    estimación de la imagen limpia.
    '''
    # TODO: completar
    raise NotImplementedError("Completar limpiar_ruido_periodico")

In [ ]:
def _test_limpiar_ruido_periodico(n_pruebas=6):
    peor_mae = 0.0
    ok_total = True

    for prueba in range(n_pruebas):
        ruidosa = agregar_ruido_periodico(foto, seed=prueba)
        try:
            limpia = limpiar_ruido_periodico(ruidosa)
        except NotImplementedError:
            print('❌ La función todavía no está implementada.')
            return False

        mae = np.abs(foto - limpia).mean()
        peor_mae = max(peor_mae, mae)
        estado = '✅' if mae < 0.02 else '❌'
        if estado == '❌':
            ok_total = False
        print(f'{estado} Prueba {prueba+1}: MAE = {mae:.4f}')

    if not ok_total:
        print(f'\nRevisen la implementación, peor MAE observado: {peor_mae:.4f}')
        return False

    print('\n🎉 ¡Todas las pruebas pasaron!')

    ruidosa_demo = agregar_ruido_periodico(foto, seed=0)
    limpia_demo = limpiar_ruido_periodico(ruidosa_demo)
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(foto, cmap='gray', vmin=0, vmax=1); axes[0].set_title('Original'); axes[0].axis('off')
    axes[1].imshow(ruidosa_demo, cmap='gray', vmin=0, vmax=1); axes[1].set_title('Con ruido'); axes[1].axis('off')
    axes[2].imshow(limpia_demo, cmap='gray', vmin=0, vmax=1); axes[2].set_title('Limpia'); axes[2].axis('off')
    fig.tight_layout(); plt.show()

    return True

_ = _test_limpiar_ruido_periodico()

## Yendo más allá: artefactos más complejos

El ruido periódico "de manual" que usamos hasta acá es una sinusoide pura, con exactamente dos picos en el espectro. Pero hay artefactos más comunes en la práctica que no son tan limpios, por ejemplo, **bandas periódicas** (rayas horizontales o verticales con bordes duros, típicas de sensores con líneas de lectura defectuosas o interferencia de la fuente de alimentación). Una onda cuadrada no es una sola frecuencia: tiene una frecuencia fundamental *y todos sus armónicos impares*, cada uno más débil que el anterior. En el espectro, en vez de un par de puntos, vemos una **línea entera** de picos.

La buena noticia: `detectar_picos_periodicos` no sabe ni le importa cuántos picos hay, evalúa cada píxel del espectro por su cuenta. El mismo método debería funcionar sin cambios.

*(Esta celda usa `detectar_picos_periodicos` y `eliminar_frecuencias`, así que necesitan haber resuelto los Ejercicios 4 y 5 para que funcione.)*

In [ ]:
from scipy import signal

y = np.arange(foto.shape[0])
bandas = 0.06 * signal.square(2*np.pi*y/16)   # banda periodica cada 16 filas
foto_con_bandas = foto + bandas[:, None]

magnitud_bandas = np.abs(np.fft.fftshift(np.fft.fft2(foto_con_bandas)))
mascara_bandas = detectar_picos_periodicos(magnitud_bandas)
print('Píxeles detectados:', mascara_bandas.sum(), 'ya no son 2, son todos los armónicos de la onda cuadrada')

limpia_bandas = eliminar_frecuencias(foto_con_bandas, mascara_bandas)

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
axes[0,0].imshow(foto, cmap='gray', vmin=0, vmax=1); axes[0,0].set_title('Original'); axes[0,0].axis('off')
axes[0,1].imshow(foto_con_bandas, cmap='gray', vmin=0, vmax=1); axes[0,1].set_title('Con bandas'); axes[0,1].axis('off')
axes[0,2].imshow(limpia_bandas, cmap='gray', vmin=0, vmax=1); axes[0,2].set_title('Limpia'); axes[0,2].axis('off')
axes[1,0].imshow(np.log1p(magnitud_bandas), cmap='magma', **ESCALA_RUIDO); axes[1,0].set_title('Espectro (con bandas)'); axes[1,0].axis('off')
axes[1,1].imshow(mascara_bandas, cmap='gray'); axes[1,1].set_title(f'Máscara ({mascara_bandas.sum()} píxeles)'); axes[1,1].axis('off')
axes[1,2].imshow(np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(limpia_bandas)))), cmap='magma', **ESCALA_RUIDO); axes[1,2].set_title('Espectro (limpia)'); axes[1,2].axis('off')
fig.tight_layout(); plt.show()

print('MAE con bandas:', np.abs(foto - foto_con_bandas).mean())
print('MAE limpia:', np.abs(foto - limpia_bandas).mean())

La máscara ya no son un par de puntos: es una **línea entera** de picos a lo largo de un eje, literalmente la forma que tiene, en frecuencia, un artefacto de bandas horizontales. Y sin cambiar una línea de código, la misma `detectar_picos_periodicos` + `eliminar_frecuencias` los encuentra y los limpia a todos juntos.

## Ejercicio 7 — El telescopio averiado, segunda parte

Malas noticias desde la nave: el diagnóstico original era incompleto. Además del kernel desconocido que difumina la óptica, el equipo de electrónica confirmó que el sensor agrega **ruido periódico** (interferencia de la fuente de alimentación) a cada lote, *después* de que la luz ya pasó por la óptica borrosa. El modelo completo de la degradación es:

$$ g = (f * h) + n $$

con $h$ el kernel óptico desconocido (como en el Ejercicio 3) y $n$ el ruido periódico desconocido (como en el Ejercicio 6).

**¿En qué orden conviene deshacer esto?**

Les dejamos armada `telescopio_averiado(lote_de_fotos, seed=None)`: funciona como `telescopio`, pero además le agrega a todo el lote el mismo ruido periódico (una interferencia por lote, igual que el kernel).

**La consigna:** implementen `reparar_lote_completo(fotos, seed=None)`, con la misma estructura que `reparar_lote` del Ejercicio 3 (coleen la delta, recuperen el kernel), pero ahora también tienen que lidiar con el ruido en *todas* las imagenes.

In [ ]:
def telescopio_averiado(lote_de_fotos, seed=None):
    '''
    Versión más realista del telescopio: además del kernel desconocido que
    difumina la óptica, el sensor agrega ruido periódico (interferencia
    electrónica) DESPUÉS del borroneo, el mismo ruido para todo el lote,
    igual que el kernel.
    '''
    rng = np.random.default_rng(seed)
    semilla_kernel = int(rng.integers(0, 2**31))
    semilla_ruido = int(rng.integers(0, 2**31))
    kernel_secreto = kernel_aleatorio(seed=semilla_kernel)
    borrosas = [convolucion_frecuencia(imagen, kernel_secreto) for imagen in lote_de_fotos]
    ruido = agregar_ruido_periodico(np.zeros_like(lote_de_fotos[0]), seed=semilla_ruido)
    return [imagen + ruido for imagen in borrosas]


def reparar_lote_completo(fotos, seed=None):
    '''
    Recibe una lista de imágenes 'fotos' (todas del mismo tamaño), las hace
    pasar por 'telescopio_averiado' junto con alguna imagen adicional que
    revele el kernel, y devuelve una lista del mismo largo con las
    versiones recuperadas de cada imagen original.

    'seed' es opcional y hay que pasárselo a 'telescopio_averiado' tal cual.
    '''
    # TODO: completar
    raise NotImplementedError("Completar reparar_lote_completo")

In [ ]:
def _test_reparar_lote_completo(n_pruebas=8):
    fotos = [img_1, img_2, img_3]
    ok_total = True
    peor_mae = 0.0

    for prueba in range(n_pruebas):
        try:
            recuperadas = reparar_lote_completo(fotos)
        except NotImplementedError:
            print('❌ La función todavía no está implementada.')
            return False

        if len(recuperadas) != len(fotos):
            print(f'❌ Prueba {prueba+1}: se esperaban {len(fotos)} imágenes recuperadas, llegaron {len(recuperadas)}.')
            ok_total = False
            continue

        maes = [np.abs(orig - rec).mean() for orig, rec in zip(fotos, recuperadas)]
        peor_mae = max(peor_mae, max(maes))
        estado = '✅' if max(maes) < 0.08 else '❌'
        if estado == '❌':
            ok_total = False
        print(f'{estado} Prueba {prueba+1}: MAE máximo entre las 3 fotos = {max(maes):.4f}')

    if not ok_total:
        print(f'\nRevisen la implementación, peor MAE observado: {peor_mae:.4f}')
        return False

    print('\n🎉 ¡Todas las pruebas pasaron! (kernel Y ruido distintos en cada una)')

    # Diagnóstico completo con semilla fija, para poder mostrar también el
    # kernel y el ruido reales (misma derivación interna que telescopio_averiado).
    SEMILLA_DEMO = random.randint(1, 100)
    rng_semilla = np.random.default_rng(SEMILLA_DEMO)
    semilla_kernel = int(rng_semilla.integers(0, 2**31))
    semilla_ruido  = int(rng_semilla.integers(0, 2**31))
    kernel_real = kernel_aleatorio(seed=semilla_kernel)
    ruido_real  = agregar_ruido_periodico(np.zeros_like(fotos[0]), seed=semilla_ruido)
    modificadas = telescopio_averiado(fotos, seed=SEMILLA_DEMO)
    recuperadas_demo = reparar_lote_completo(fotos, seed=SEMILLA_DEMO)

    fig, axes = plt.subplots(len(fotos), 5, figsize=(15, 3.3*len(fotos)))
    for i, (orig, mod, rec) in enumerate(zip(fotos, modificadas, recuperadas_demo)):
        axes[i,0].imshow(orig, cmap='gray', vmin=0, vmax=1); axes[i,0].axis('off')
        axes[i,1].imshow(kernel_real, cmap='gray', vmin=min(0, kernel_real.min()), vmax=kernel_real.max())
        axes[i,1].set_xticks([]); axes[i,1].set_yticks([])
        axes[i,2].imshow(ruido_real, cmap='gray'); axes[i,2].axis('off')
        axes[i,3].imshow(mod, cmap='gray', vmin=0, vmax=1); axes[i,3].axis('off')
        axes[i,4].imshow(rec, cmap='gray', vmin=0, vmax=1); axes[i,4].axis('off')
    for col, titulo in enumerate(['Original', 'Kernel', 'Ruido', 'Modificada', 'Recuperada']):
        axes[0, col].set_title(titulo)
    fig.tight_layout(); plt.show()

    return True

_ = _test_reparar_lote_completo()